# 01 - Train the mock-interviewer (QLoRA on Qwen2.5-1.5B-Instruct)
**Before you run anything:** Runtime -> Change runtime type -> **T4 GPU**. Then run the cells top to bottom.
Cell 3 asks you to upload `interview_bot.zip`. Expected time on a T4 is roughly 30-60 minutes (estimate, not measured).

In [ ]:
!nvidia-smi -L
!pip install -q -U transformers peft bitsandbytes accelerate datasets gradio

In [ ]:
# Upload interview_bot.zip when prompted
from google.colab import files
up = files.upload()
!unzip -o -q interview_bot.zip -d /content
%cd /content/interview_bot

In [ ]:
# Data is generated by rules in core.py (no LLM involved). Regenerating gives the same files (fixed seed).
!python gen_data.py
!head -c 1500 data/train.jsonl

In [ ]:
import torch, json, random
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU"
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map={"": 0})
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

### Tokenise with loss only on the interviewer's turns
We render each conversation with the model's chat template and keep labels only for assistant messages
(system prompt, STATE lines and candidate answers are masked with -100).

In [ ]:
MAXLEN = 1536

def render(msgs, gen=False):
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=gen)

def encode(msgs):
    full = render(msgs)
    spans = [(len(render(msgs[:i], gen=True)), len(render(msgs[:i + 1])))
             for i, m in enumerate(msgs) if m["role"] == "assistant"]
    enc = tok(full, add_special_tokens=False, return_offsets_mapping=True, truncation=True, max_length=MAXLEN)
    ids = enc["input_ids"]
    labels = [-100] * len(ids)
    for k, (a, b) in enumerate(enc["offset_mapping"]):
        if b > a and any(a >= s and b <= e for s, e in spans):
            labels[k] = ids[k]
    return {"input_ids": ids, "labels": labels}

def load(p):
    return [json.loads(l) for l in open(p)]

train = [encode(r["messages"]) for r in load("data/train.jsonl")]
val = [encode(r["messages"]) for r in load("data/val.jsonl")]
print("train", len(train), "val", len(val), "max tokens", max(len(x["input_ids"]) for x in train))
# sanity check: the supervised text should be ONLY interviewer replies
print(tok.decode([t for t in train[0]["labels"] if t != -100])[:600])

In [ ]:
def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    return {
        "input_ids": torch.tensor([b["input_ids"] + [pad] * (n - len(b["input_ids"])) for b in batch]),
        "attention_mask": torch.tensor([[1] * len(b["input_ids"]) + [0] * (n - len(b["input_ids"])) for b in batch]),
        "labels": torch.tensor([b["labels"] + [-100] * (n - len(b["labels"])) for b in batch]),
    }

args = TrainingArguments(
    output_dir="out", per_device_train_batch_size=2, gradient_accumulation_steps=8, num_train_epochs=2,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.05, fp16=True, logging_steps=5,
    save_strategy="no", report_to="none", gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False}, optim="paged_adamw_8bit", remove_unused_columns=False)
model.config.use_cache = False
trainer = Trainer(model=model, args=args, train_dataset=train, eval_dataset=val, data_collator=collate)
trainer.train()
print("validation:", trainer.evaluate())

In [ ]:
model.save_pretrained("adapter")
tok.save_pretrained("adapter")

import matplotlib.pyplot as plt
logs = [l for l in trainer.state.log_history if "loss" in l]
plt.plot([l["step"] for l in logs], [l["loss"] for l in logs])
plt.xlabel("step"); plt.ylabel("training loss"); plt.title("Training loss"); plt.grid(alpha=.3)
plt.savefig("loss_curve.png", dpi=150); plt.show()

In [ ]:
# Save a copy so a Colab disconnect does not lose your work
import shutil
shutil.make_archive("adapter", "zip", "adapter")
from google.colab import drive
drive.mount("/content/drive")
!cp -r adapter /content/drive/MyDrive/interview_adapter
!cp loss_curve.png /content/drive/MyDrive/
print("saved. Now open 02_eval.ipynb (run it in this same session).")